# Ficheros `.env` en Java

Un fichero `.env` es un fichero de texto plano que almacena variables de configuración en formato `CLAVE=VALOR`. Se utiliza habitualmente para guardar información sensible (contraseñas, claves de API, URL de bases de datos, etc.) fuera del código fuente.

## ¿Por qué usar ficheros `.env`?

- **Seguridad**: evitan que datos sensibles queden expuestos en el código fuente o en el historial de Git.
- **Portabilidad**: permiten configurar la aplicación de forma diferente en cada entorno (desarrollo, pruebas, producción) sin modificar el código.
- **Mantenimiento**: centralizan la configuración en un único lugar.

## Estructura de un fichero `.env`

```
# Esto es un comentario
DB_HOST=localhost
DB_PORT=5432
DB_USER=admin
DB_PASS=s3cr3t
API_KEY=abc123xyz
```

> **Importante**: el fichero `.env` **nunca debe subirse** al repositorio. Añádelo siempre a `.gitignore`. En su lugar, puedes proporcionar un fichero de ejemplo (`.env.example`) con las claves pero sin los valores sensibles.

## Uso de la librería `dotenv-java`

La librería [`dotenv-java`](https://github.com/cdimascio/dotenv-java) simplifica la carga del fichero `.env` y gestiona casos como comentarios, valores con espacios o fichero ausente. Para usarla hay que añadir la dependencia al proyecto.

### Con Maven (`pom.xml`)

```xml
<dependency>
    <groupId>io.github.cdimascio</groupId>
    <artifactId>dotenv-java</artifactId>
    <version>3.2.0</version>
</dependency>
```

### Ubicación del fichero `.env` en un proyecto Maven

El fichero `.env` debe colocarse en la **raíz del proyecto**, al mismo nivel que el `pom.xml`:

```
mi-proyecto/
├── pom.xml
├── .env          ← aquí
├── .gitignore
└── src/
    └── main/
        └── java/...
```

`dotenv-java` busca el `.env` en el directorio de trabajo actual, que suele ser la raíz del proyecto. Si el fichero no se encuentra, la librería lanzará una excepción.

In [ ]:
// Requiere la dependencia dotenv-java en el proyecto
import io.github.cdimascio.dotenv.Dotenv;

Dotenv dotenv = Dotenv.load(); // busca .env en la raíz del proyecto

String host = dotenv.get("DB_HOST");
String user = dotenv.get("DB_USER");
String pass = dotenv.get("DB_PASS");

System.out.println("Conectando a " + host + " como " + user);

### Opciones de configuración

Se puede personalizar el comportamiento mediante `Dotenv.configure()`:

| Método | Descripción |
|---|---|
| `.directory("ruta")` | Carpeta donde buscar el fichero `.env` |
| `.filename("nombre")` | Nombre alternativo del fichero |
| `.ignoreIfMissing()` | No lanza excepción si el fichero no existe |
| `.ignoreIfMalformed()` | Ignora líneas con formato incorrecto |

In [ ]:
Dotenv dotenv = Dotenv.configure()
    .directory("./config")    // el .env está en la carpeta config/
    .filename(".env.local")   // nombre alternativo del fichero
    .ignoreIfMissing()        // no falla si no existe el fichero
    .load();

// El segundo parámetro es el valor por defecto si la clave no existe
String apiKey = dotenv.get("API_KEY", "sin-clave");
System.out.println("API_KEY: " + apiKey);

## Uso práctico: conexión a una base de datos

Un caso de uso muy habitual es almacenar los datos de conexión a la base de datos en el fichero `.env` para no escribirlos directamente en el código.

**Fichero `.env`:**
```
DB_HOST=jdbc:postgresql://localhost:5432/mibasededatos
DB_USER=admin
DB_PASS=s3cr3t
```

In [ ]:
import java.sql.Connection;
import java.sql.DriverManager;
import io.github.cdimascio.dotenv.Dotenv;

Dotenv dotenv = Dotenv.load();

String host  = dotenv.get("DB_HOST");
String user = dotenv.get("DB_USER");
String pass = dotenv.get("DB_PASS");

try (Connection conn = DriverManager.getConnection(host, user, pass)) {
    System.out.println("Conexión establecida correctamente");
} catch (Exception e) {
    System.err.println("Error al conectar: " + e.getMessage());
}

## Buenas prácticas

1. **Añadir `.env` al `.gitignore`** para que nunca se suba al repositorio:
   ```
   .env
   ```

2. **Incluir un fichero `.env.example`** con las mismas claves pero sin valores reales, para que otros desarrolladores sepan qué variables necesitan configurar:
   ```
   DB_URL=
   DB_USER=
   DB_PASS=
   API_KEY=
   ```

3. **No usar `.env` en producción**: en entornos de producción es preferible usar variables de entorno del sistema operativo o un gestor de secretos (AWS Secrets Manager, HashiCorp Vault, etc.).

4. **Un valor por clave**: si el valor contiene el carácter `=`, hay que entrecomillarlo para que la lectura sea correcta:
   ```
   TOKEN="abc=def=ghi"
   ```